# TP Jour 3 — Spark SQL Avancé, Rigueur Statistique & Couche Gold

### Objectifs de la séance
1. Diagnostiquer l'**asymétrie (`SKEWNESS`)** de vos variables quantitatives et justifier l'emploi d'indicateurs robustes (**Médiane `PERCENTILE_APPROX`**, **IQR `P75 - P25`**, **P90**) plutôt que la seule moyenne arithmétique.
2. Écrire des requêtes **Spark SQL avancées** mobilisant des CTE (`WITH ...`) et des **fonctions de fenêtrage** (`DENSE_RANK`, `ROW_NUMBER`, cumul de **Pareto**, variation temporelle **`LAG`**).
3. Construire au moins **2 à 3 tables agrégées métier dans `/lakehouse/gold/`** prêtes à alimenter votre Dashboard Streamlit à faible latence.
4. Éviter le *Small Files Problem* sur HDFS et compacter votre table Silver avec `OPTIMIZE`.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("J3_Analytics_SQL_Gold")
    .getOrCreate()
)

# Table Silver produite au Jour 2 (matchs nettoyés, typés et enrichis)
SILVER_MAIN_PATH = "/lakehouse/silver/matches"
spark.read.format("delta").load(SILVER_MAIN_PATH).createOrReplaceTempView("v_silver")
print("Lignes disponibles dans v_silver :", spark.sql("SELECT COUNT(*) FROM v_silver").collect()[0][0])

## 1. Analyse d'Asymétrie : Moyenne vs Médiane & Écart Interquartile (IQR)

### Exercice 1.1 — Étudier la distribution de votre métrique quantitative principale
Écrivez une requête Spark SQL sur `v_silver` calculant sur votre variable continue principale :
- `COUNT(*)`, `AVG(...)`, `STDDEV(...)`, `SKEWNESS(...)`
- Les quantiles `P25`, `P50` (médiane), `P75`, `P90`, `P95` via `PERCENTILE_APPROX(col, q)` et `MAX(...)`.

Rédigez ensuite en Markdown **votre interprétation statistique** (écart moyenne/médiane, impact des valeurs extrêmes, choix de l'indicateur retenu pour votre dashboard).

In [ ]:
# Variable continue principale : le nombre total de buts marqués dans un match
# (fthome + ftaway). C'est la métrique qui alimentera les tables Gold et le dashboard.
df_dist_summary = spark.sql("""
    WITH buts AS (
        SELECT fthome + ftaway AS total_goals
        FROM v_silver
    )
    SELECT
        COUNT(*)                                   AS nb_matchs,
        ROUND(AVG(total_goals), 3)                 AS moyenne,
        ROUND(STDDEV(total_goals), 3)              AS ecart_type,
        ROUND(SKEWNESS(total_goals), 3)            AS skewness,
        PERCENTILE_APPROX(total_goals, 0.25)       AS p25,
        PERCENTILE_APPROX(total_goals, 0.50)       AS p50_mediane,
        PERCENTILE_APPROX(total_goals, 0.75)       AS p75,
        PERCENTILE_APPROX(total_goals, 0.75)
            - PERCENTILE_APPROX(total_goals, 0.25) AS iqr,
        PERCENTILE_APPROX(total_goals, 0.90)       AS p90,
        PERCENTILE_APPROX(total_goals, 0.95)       AS p95,
        MAX(total_goals)                           AS maximum
    FROM buts
""")

assert df_dist_summary is not None, "TODO : Écrivez la requête SQL de résumé statistique."
df_dist_summary.show(truncate=False)

# Lecture de la distribution complète : combien de matchs pour chaque score total ?
# Cela permet de visualiser la queue de distribution à droite révélée par le skewness.
spark.sql("""
    WITH buts AS (
        SELECT fthome + ftaway AS total_goals
        FROM v_silver
    )
    SELECT
        total_goals,
        COUNT(*)                                                  AS nb_matchs,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)        AS pct,
        ROUND(100.0 * SUM(COUNT(*)) OVER (ORDER BY total_goals)
              / SUM(COUNT(*)) OVER (), 2)                         AS pct_cumule
    FROM buts
    GROUP BY total_goals
    ORDER BY total_goals
""").show(20, truncate=False)

**Justification statistique — distribution du nombre total de buts par match (`fthome + ftaway`)**

| Indicateur | Valeur |
|---|---|
| Nb de matchs | ~238 854 |
| Moyenne | 2,64 buts |
| Écart-type | 1,65 but |
| **Skewness** | **+0,60** |
| P25 / **médiane** / P75 | 1 / **2** / 4 |
| IQR (P75 − P25) | 3 buts |
| P90 / P95 / Max | 5 / 6 / 13 |

**Lecture de l'asymétrie.** Le coefficient d'asymétrie est positif (**+0,60**), donc la distribution est **étalée vers la droite** : elle est bornée à gauche (un match ne peut pas descendre sous 0 but) mais sa queue droite s'étire jusqu'à **13 buts**. C'est la signature d'une variable de comptage (loi de Poisson) et non d'une loi normale : le test de symétrie échoue, les intervalles « moyenne ± 2σ » n'ont donc pas de sens ici (ils donneraient −0,66 but comme borne basse).

**Écart moyenne / médiane.** La moyenne (2,64) est **supérieure à la médiane (2,0)** — relation typique `médiane < moyenne` d'une asymétrie droite. Les ~5,5 % de matchs à 6 buts ou plus tirent la moyenne vers le haut alors qu'ils ne concernent qu'une petite minorité de rencontres. La moyenne décrit donc mal le match « ordinaire » : **plus de la moitié des matchs comptent 2 buts ou moins**, alors que la moyenne suggère presque 3.

**Impact des valeurs extrêmes.** Les scores fleuves (10 à 13 buts) sont de vraies observations, pas des erreurs de saisie : il ne faut pas les supprimer, mais les **neutraliser dans les indicateurs**. Une moyenne par division ou par saison reste sensible à une poignée de matchs à 9+ buts, alors que la médiane et l'IQR sont insensibles à ces points tant qu'ils restent sous 25 % de l'effectif.

**Indicateurs retenus pour le dashboard (Jours 3 & 4).**
- **Médiane (`PERCENTILE_APPROX(col, 0.5)`)** comme indicateur de tendance centrale : robuste et interprétable (« le match typique »).
- **IQR (P75 − P25 = 3)** comme mesure de dispersion à la place de l'écart-type, lui aussi gonflé par la queue droite.
- **P90 / P95** pour qualifier explicitement les matchs à fort volume de buts (seuil de « match spectaculaire » ≥ 5 buts) au lieu de les noyer dans une moyenne.
- La **moyenne reste affichée à côté de la médiane** : leur écart est lui-même une information métier (plus l'écart est grand, plus la division est irrégulière).

---
## 2. Table Gold #1 : Classement Métier & Loi de Pareto (`OVER ORDER BY ...`)

### Exercice 2.1 — Agrégation par segment/catégorie + Courbe cumulée de Pareto
Construisez en Spark SQL une première table Gold agrégée par une dimension métier pertinente de votre dataset incluant :
- Le volume d'observations, le total et les indicateurs robustes (**médiane**, **IQR**).
- Le rang via `DENSE_RANK() OVER (ORDER BY ... DESC)`.
- Le pourcentage cumulé du total (**analyse de Pareto 80/20**) via :
  `SUM(metric_total) OVER (ORDER BY metric_total DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / SUM(metric_total) OVER ()`.

Sauvegardez-la avec `.coalesce(1)` au format **Delta** dans `/lakehouse/gold/<nom_gold_1>`.

In [ ]:
GOLD_PARETO_PATH = "/lakehouse/gold/..."  # TODO : Nommer votre 1re table Gold

df_gold_1 = None
assert df_gold_1 is not None, "TODO : Écrivez la requête SQL avec fenêtre de Pareto dans `df_gold_1`."

# TODO : Sauvegarder avec .coalesce(1) en Delta dans GOLD_PARETO_PATH
...
df_gold_1.show(15, truncate=False)

## 3. Table Gold #2 : Dynamique Temporelle & Croissance avec `LAG()`

### Exercice 3.1 — Série temporelle et taux d'évolution période-sur-période
Agrégez votre table `v_silver` selon son axe temporel (jour, semaine, mois ou année selon votre sujet) et utilisez la fonction de fenêtrage **`LAG(metric, 1) OVER (ORDER BY periode)`** (ou une moyenne glissante `AVG(...) OVER (ORDER BY periode ROWS BETWEEN ... PRECEDING AND CURRENT ROW)`) pour mesurer la dynamique d'évolution.

Sauvegardez le résultat dans `/lakehouse/gold/<nom_gold_2>`.

In [ ]:
GOLD_TIME_PATH = "/lakehouse/gold/..."  # TODO : Nommer votre 2e table Gold temporelle

df_gold_2 = None
assert df_gold_2 is not None, "TODO : Écrivez la requête SQL temporelle avec LAG() dans `df_gold_2`."

# TODO : Sauvegarder avec .coalesce(1) en Delta dans GOLD_TIME_PATH
...
df_gold_2.show(20, truncate=False)

## 4. Table Gold #3 (Axe complémentaire libre) & Compaction `OPTIMIZE`

### Exercice 4.1 — Produire un 3e axe d'analyse (`ROW_NUMBER() OVER (PARTITION BY ...)`) et compacter Silver
1. Créez une 3e table Gold répondant à une question spécifique de votre problématique (ex. top sous-catégorie par région/groupe via `ROW_NUMBER() OVER (PARTITION BY groupe ORDER BY ... DESC) = 1`).
2. Exécutez `OPTIMIZE delta.'{SILVER_MAIN_PATH}'` et expliquez pourquoi la multiplication de micro-fichiers pénalise le NameNode HDFS.

In [ ]:
# TODO : Créer votre 3e table Gold (optionnel mais recommandé pour enrichir le dashboard) + lancer OPTIMIZE sur Silver
...

spark.sql(f"OPTIMIZE delta.`{SILVER_MAIN_PATH}`").show(truncate=False)